# 11 - Trained on BERT-Attack only: does it hold against TextFooler and PWWS?

The mirror image of notebook 10. The same recipe is trained on **BERT-Attack examples only** and tested on all three attacks. This is the strongest cross-attack test we can run: BERT-Attack takes its replacement words from BERT's own masked-word guesses, while TextFooler and PWWS take them from counter-fitted word vectors. The defense below has never seen a single substitution of the kind those two attacks make.

**What counts as "never seen".** No TextFooler or PWWS example is used anywhere: not in training BERT, not in training the encoder, not in choosing the encoder's epoch. To rule out indirect use too, the BERT-Attack examples are generated only against:
- vanilla BERT (new runs on training sentences);
- this notebook's own BERT (a second round).

None of the earlier encoders is used, because they were trained on TextFooler or PWWS examples, or on synonym lists.

**Recipe:** identical to notebooks 07b and 10: the same starting BERT and noise, 2 + 1 epochs of BERT training, the paper's Table 6 GAN, and the encoder's epoch chosen on attacked *validation* sentences (BERT-Attack only here). It uses the same clean-accuracy check as notebook 10: an encoder more than 1 point below BERT's own clean validation accuracy is rejected and retrained with the next seed. Nothing is chosen on test; one test run with all three attacks.

**Cost (L4, rough):** about 4-4.5 hours, including about 1.5 hours of BERT-Attack runs on training sentences. Everything is cached on Drive and resumes after a disconnect. Run after 07b; then re-run 09.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, json, torch, pandas as pd, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # same folder as notebooks 00-10
N_TEST = 1000
N_FIRST, N_ROUND = 3000, 2000      # training sentences attacked against vanilla BERT, then against this notebook's BERT
ALL = ('textfooler', 'bertattack', 'pwws')
ENC_EPOCHS = 30
W_DEN = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=0.0, w_cls=0.0)   # as in 07b and 10
W_ADV = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=1.0, w_cls=1.0)

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
vanilla = T.Victim(lab, mode='none')

# Start exactly where 07b and 10 started
START = json.load(open(lab._path('finetuned_advbert1/DONE')))['init']
RATE = (0.1, 0.3) if START == 'base' else tuple(json.load(open(lab._path('nb07_choice.json')))['rate'])
print('starting BERT, as in 07b:', START, '| noise rate:', RATE)
noise = T.NoiseAugmenter(lab.train_ds['text'], rate=RATE, seed=1)

def texts_from_logs(names):
    # (clean, attacked, label) of every successful attack in these exact log files
    out, seen = [], set()
    for name in names:
        p = lab._path(name)
        assert os.path.exists(p), f'missing {p}'
        for r in map(json.loads, open(p)):
            if r['status'] == 'S' and (r['text'], r['adv_text']) not in seen:
                seen.add((r['text'], r['adv_text']))
                out.append((r['text'], r['adv_text'], r['label']))
    return out

# BERT-Attack on TRAINING sentences against vanilla BERT (new runs)
T.adversarial_pairs(lab, vanilla, 'vanilla', 'bertattack', N_FIRST, 51)
BA_LOGS = [f'advpairs_vanilla_bertattack_n{N_FIRST}_s51.jsonl']
VAL_LOGS = ['advpairs_val_vanilla_bertattack_n500_s12.jsonl']   # attacked VALIDATION sentences, for the epoch choice
ba_tr = texts_from_logs(BA_LOGS)
print(len(ba_tr), 'BERT-Attack-attacked training sentences')

## Step 1: train BERT on clean, noisy and BERT-Attack-attacked training sentences

In [ ]:
lab.finetune_robust('babert1', init=START, epochs=2, lr=1e-5, noise_fn=noise, adv_texts=ba_tr)

## Step 2: attack this BERT with BERT-Attack on fresh training sentences, then train once more

In [ ]:
lab.use_model('babert1')
T.adversarial_pairs(lab, T.Victim(lab, mode='none'), 'babert1', 'bertattack', N_ROUND, 52)
ba_tr2 = texts_from_logs(BA_LOGS + [f'advpairs_babert1_bertattack_n{N_ROUND}_s52.jsonl'])
print(len(ba_tr2), 'BERT-Attack-attacked training sentences in total')
lab.finetune_robust('babert2', init='babert1', epochs=1, lr=1e-5, noise_fn=noise, adv_texts=ba_tr2)

## Step 3: the projection (Table 6 GAN + learned encoder, with the clean-accuracy check)

In [ ]:
lab.use_model('babert2')
Eb_tr, _ = lab.embeddings('train', 'babert2')
Eb_va, _ = lab.embeddings('val', 'babert2')
Eb_te, _ = lab.embeddings('test', 'babert2')
Eb_noise, _ = lab.embeddings('train', 'babert2', augment_fn=noise, name='train_noise')
gan_b = lab.get_gan('babert2', Eb_tr, Eb_va)
head_b = lab.model.classifier
adv_b = (T.embed_texts(lab, [r[1] for r in ba_tr2]), T.embed_texts(lab, [r[0] for r in ba_tr2]),
         torch.tensor([r[2] for r in ba_tr2]))
val_rows = texts_from_logs(VAL_LOGS)
val_adv_b = (T.embed_texts(lab, [r[1] for r in val_rows]), torch.tensor([r[2] for r in val_rows]))

def get_enc(name, **kw):
    # Train an encoder on this BERT, or load it from Drive
    p = lab._path(f'projenc_{name}_L{gan_b.latent_dim}C{gan_b.code_dim}.pt')
    e = T.ProjEncoder(gan_b.latent_dim, gan_b.code_dim).to(T.DEV)
    if os.path.exists(p):
        e.load_state_dict(torch.load(p)); print('loaded', name)
        return e.eval()
    print('training', name)
    e = T.train_proj_encoder(gan_b, Eb_tr, Y_tr, Eb_va, Y_va, E_aug=Eb_noise, head=head_b, epochs=ENC_EPOCHS, **kw)
    torch.save(e.state_dict(), p)
    return e

enc_den_b = get_enc('ba_enc_den', **W_DEN)

# Clean-accuracy check (as in notebook 10): keep the first seed whose encoder stays within 1 point of BERT's own
with torch.no_grad():
    bert_clean = (head_b(Eb_va.to(T.DEV)).argmax(-1).cpu() == Y_va).float().mean().item()
def proj_clean(e):
    with torch.no_grad():
        th = T.project(gan_b, Eb_va.to(T.DEV), enc=e, mode='encoder')[0]
        return (head_b(th).argmax(-1).cpu() == Y_va).float().mean().item()
for seed in (42, 1, 2, 3):
    name = 'ba_enc1' if seed == 42 else f'ba_enc1_s{seed}'
    enc1_b = get_enc(name, adv=adv_b, init=enc_den_b, select='best', val_adv=val_adv_b, seed=seed, **W_ADV)
    acc = proj_clean(enc1_b)
    print(f'{name}: projected clean val acc {acc:.4f} (BERT alone {bert_clean:.4f})')
    if acc >= bert_clean - 0.01:
        break
else:
    raise RuntimeError('no seed kept clean accuracy: send the log')
TAG = 'ba_only_enc1' if seed == 42 else f'ba_only_enc1_s{seed}'

## The one test run: seen attack (BERT-Attack) and unseen attacks (TextFooler, PWWS)

In [ ]:
lab.use_model('babert2')
v = T.Victim(lab, gan=gan_b, enc=enc1_b, mode='tmd', proj_mode='encoder')
T.evaluate(lab, v, TAG, Eb_te, Y_te, attacks=ALL, n=N_TEST, need_log=True)
display(T.compare(lab, TAG, baseline_tag='baseline', n=N_TEST))
display(pd.DataFrame([T.mcnemar(lab, 'baseline', TAG, a, N_TEST) for a in ALL]))

# The train-on-A, test-on-B matrix
P = dict(zip(T.METRICS, T.PAPER['ag_news']['TMD']))
rows = [{'defense': 'paper: TMD', 'clean': P['clean'], 'TextFooler': P['textfooler'], 'BERT-Attack': P['bertattack'],
         'PWWS': P['pwws']}]
tf_only = next((t for t in ('tf_only_enc1_s1', 'tf_only_enc1_s2', 'tf_only_enc1_s3', 'tf_only_enc1')
                if os.path.exists(T.results_path(lab, t, N_TEST))
                and json.load(open(T.results_path(lab, t, N_TEST)))['clean']['acc'] > 90), None)
for tag, label in [('baseline', 'ours: TMD reproduced'), (tf_only, 'ours, trained on TextFooler only (10)'),
                   (TAG, 'ours, trained on BERT-Attack only (11)'), ('advbert2_enc1', 'ours, trained on all three (07b)')]:
    if tag and os.path.exists(T.results_path(lab, tag, N_TEST)):
        r = json.load(open(T.results_path(lab, tag, N_TEST)))
        rows.append({'defense': label, 'clean': round(r['clean']['acc'], 2),
                     **{T.METRIC_LABEL[a]: round(r[a]['acc'], 1) for a in ALL if a in r}})
display(pd.DataFrame(rows))
print('Row 11 never saw a TextFooler or PWWS example, in training or in choosing its epoch.')

In [ ]:
# Transfer and per-sentence worst case for this defense, saved with notebook 08's results for notebook 09
RES = lab._path('nb08_results.json')
res = json.load(open(RES)) if os.path.exists(RES) else {}
out = [r for r in res.get('worst_case', []) if r['defense'] != TAG]
for attack in ALL:
    src = [r for r in map(json.loads, open(T.examples_path(lab, 'vanilla', N_TEST, attack))) if r['status'] == 'S']
    y, texts = torch.tensor([r['label'] for r in src]), [r['adv_text'] for r in src]
    pred = torch.cat([v.scores(texts[i:i + 64]).argmax(-1).cpu() for i in range(0, len(texts), 64)])
    ok = {r['i']: bool(p) for r, p in zip(src, pred == y)}
    adapt = {r['i']: r['status'] for r in map(json.loads, open(T.examples_path(lab, TAG, N_TEST, attack)))
             if r['status'] != 'D'}
    n = len(adapt)
    out.append(dict(attack=attack, defense=TAG, adaptive=round(100 * sum(s == 'F' for s in adapt.values()) / n, 1),
                    transfer=round(100 * (pred == y).float().mean().item(), 1), transfer_n=len(src),
                    worst_case=round(100 * sum(s == 'F' and ok.get(i, True) for i, s in adapt.items()) / n, 1)))
res['worst_case'] = out
json.dump(res, open(RES, 'w'), indent=1)
display(pd.DataFrame([r for r in out if r['defense'] == TAG]))